In [2]:
print("Installing Apache Airflow... (This may take around 30-45 seconds)")
!pip install apache-airflow > /dev/null 2>&1
print("✔ Apache Airflow installed successfully!\n")

Installing Apache Airflow... (This may take around 30-45 seconds)
✔ Apache Airflow installed successfully!



In [3]:
import os
from datetime import datetime
from collections import defaultdict
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator

# --- Task 1: Create the Department File ---
def create_department_file():
    print("--- Running Task 1: create_department_file ---")
    file_path = "/tmp/departments.txt"
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    content = """IT,45000
HR,35000
Finance,50000
IT,55000
Finance,40000
HR,30000"""

    with open(file_path, "w") as f:
        f.write(content.strip())
    print(f"✔ File generated at: {file_path}")

In [5]:
def calculate_department_salary():
    print("\n--- Running Task 2: calculate_department_salary ---")
    file_path = "/tmp/departments.txt"
    dept_salaries = defaultdict(int)

    # Read and aggregate salaries
    with open(file_path, "r") as f:
        for line in f:
            if line.strip():
                dept, salary = line.strip().split(",")
                dept_salaries[dept] += int(salary)

    print("Calculated Totals:")
    for dept, total in dept_salaries.items():
        print(f"  {dept} = {total}")

    # We save these calculations temporarily to write them in Task 3
    return dept_salaries



In [6]:
def generate_department_report():
    print("\n--- Running Task 3: generate_department_report ---")

    # Re-calculating directly for pipeline simplicity
    file_path = "/tmp/departments.txt"
    report_path = "/tmp/department_report.txt"
    dept_salaries = defaultdict(int)

    with open(file_path, "r") as f:
        for line in f:
            if line.strip():
                dept, salary = line.strip().split(",")
                dept_salaries[dept] += int(salary)

    # Write the summary report file
    with open(report_path, "w") as f:
        for dept, total in dept_salaries.items():
            f.write(f"{dept} = {total}\n")

    print(f"✔ Report compiled successfully at: {report_path}")
    print("\n📋 Final Written File Content:")
    with open(report_path, "r") as f:
        print(f.read())

In [7]:
default_args = {
    'owner': 'hexaware_training',
    'start_date': datetime(2026, 1, 1),
}

with DAG(
    dag_id='exercise_7_department_salary',
    default_args=default_args,
    schedule=None,
    catchup=False
) as dag:

    task_create_file = PythonOperator(
        task_id='create_department_file',
        python_callable=create_department_file
    )

    task_calculate = PythonOperator(
        task_id='calculate_department_salary',
        python_callable=calculate_department_salary
    )

    task_report = PythonOperator(
        task_id='generate_department_report',
        python_callable=generate_department_report
    )

    # Establish pipeline sequence
    task_create_file >> task_calculate >> task_report

print("✔ Airflow DAG Definition loaded cleanly.")

✔ Airflow DAG Definition loaded cleanly.


In [8]:
print("\nTriggering DAG execution pipeline sequentially...")
create_department_file()
calculate_department_salary()
generate_department_report()


Triggering DAG execution pipeline sequentially...
--- Running Task 1: create_department_file ---
✔ File generated at: /tmp/departments.txt

--- Running Task 2: calculate_department_salary ---
Calculated Totals:
  IT = 100000
  HR = 65000
  Finance = 90000

--- Running Task 3: generate_department_report ---
✔ Report compiled successfully at: /tmp/department_report.txt

📋 Final Written File Content:
IT = 100000
HR = 65000
Finance = 90000

